# Binary rain classification CNN training

Loads the data with `Binary_data_pipeline.py`, trains the CNN and saves the best model to `best_model.keras` and the training history to `training_history.csv`.

In [ ]:
import gc
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, regularizers
from tensorflow.keras.models import Sequential
from Binary_data_pipeline import build_datasets

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
CSV_PATH       = "Split70-15-15.csv"
EPOCHS         = 50
N_CHANNELS     = 1 
BATCH_SIZE     = 32
LEARNING_RATE  = 1e-3
SEED           = 42

PATIENCE_EARLY_STOPPING = 15
PATIENCE_REDUCE_LR      = 5
LR_REDUCE_FACTOR        = 0.5
LR_MIN                  = 1e-6

In [ ]:
class GarbageCollectionCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        collected = gc.collect()
        print(f"   [GC] {collected} objetos liberados.")

In [ ]:
def ArquiteturaCNN(input_shape=(256, 256, N_CHANNELS)):
    l2_reg = regularizers.l2(0.0005)

    model = Sequential(name="ArquiteturaCNN")

    # ---------- L1 ----------
    model.add(layers.Conv2D(32, kernel_size=3, padding="same", kernel_regularizer=l2_reg, input_shape=input_shape))
    model.add(layers.LeakyReLU(alpha=0.1))
    model.add(layers.BatchNormalization())

    # ---------- L2 ----------
    model.add(layers.Conv2D(32, kernel_size=3, padding="same", kernel_regularizer=l2_reg))
    model.add(layers.LeakyReLU(alpha=0.1))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(pool_size=2))

    # ---------- L3 ----------
    model.add(layers.Conv2D(64, kernel_size=3, padding="same", kernel_regularizer=l2_reg))
    model.add(layers.LeakyReLU(alpha=0.1))
    model.add(layers.BatchNormalization())

    # ---------- L4 ----------
    model.add(layers.Conv2D(64, kernel_size=3, padding="same", kernel_regularizer=l2_reg))
    model.add(layers.LeakyReLU(alpha=0.1))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(pool_size=2))
    model.add(layers.SpatialDropout2D(0.05))

    # ---------- L5 ----------
    model.add(layers.Conv2D(128, kernel_size=3, padding="same", kernel_regularizer=l2_reg))
    model.add(layers.LeakyReLU(alpha=0.1))
    model.add(layers.BatchNormalization())

    # ---------- L6 ----------
    model.add(layers.Conv2D(128, kernel_size=3, padding="same", kernel_regularizer=l2_reg))
    model.add(layers.LeakyReLU(alpha=0.1))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(pool_size=2))
    model.add(layers.SpatialDropout2D(0.10))

    # ---------- L7 ----------
    model.add(layers.Conv2D(256, 3, padding="same", kernel_regularizer=l2_reg))
    model.add(layers.LeakyReLU(alpha=0.1))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(2))
    model.add(layers.SpatialDropout2D(0.10)) 

    # ---------- L8 ----------
    model.add(layers.Conv2D(256, 3, padding="same", kernel_regularizer=l2_reg))
    model.add(layers.LeakyReLU(alpha=0.1))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(2))
    model.add(layers.SpatialDropout2D(0.15))

    # ---------- L9: Output ----------
    model.add(layers.Flatten())
    model.add(layers.Dense(128, kernel_regularizer=l2_reg))
    model.add(layers.LeakyReLU(alpha=0.1))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(1, activation="sigmoid"))

    return model

In [ ]:
train_ds, val_ds, test_ds = build_datasets(CSV_PATH, n_channels=N_CHANNELS, batch_size=BATCH_SIZE)

model = ArquiteturaCNN(input_shape=(256, 256, N_CHANNELS))
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc"),
    ],
)
model.summary()

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath="best_model.keras", monitor="val_loss",
        mode="min", save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", mode="min",
        patience=PATIENCE_EARLY_STOPPING,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", mode="min",
        factor=LR_REDUCE_FACTOR, patience=PATIENCE_REDUCE_LR,
        min_lr=LR_MIN, verbose=1
    ),
    tf.keras.callbacks.CSVLogger(filename="training_history.csv"),
    GarbageCollectionCallback(),
]

In [ ]:
train_ds

In [ ]:
# Training
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
n_epochs_run  = len(history.history["loss"])
best_val_loss = min(history.history["val_loss"])
best_val_auc  = max(history.history["val_auc"])

print(f"Épocas executadas: {n_epochs_run} / {EPOCHS}")
print(f"Melhor val_loss:   {best_val_loss:.4f}")
print(f"Melhor val_auc:    {best_val_auc:.4f}")
print("Modelo salvo em:   best_model.keras")
print("Histórico em:      training_history.csv")

In [ ]:
# Learning curves

# Loss
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.title("Loss")
plt.xlabel("Época")
plt.legend()
plt.tight_layout()
plt.show()

# AUC
plt.figure(figsize=(8, 5))
plt.plot(history.history["auc"], label="train")
plt.plot(history.history["val_auc"], label="val")
plt.title("AUC")
plt.xlabel("Época")
plt.legend()
plt.tight_layout()
plt.show()

# Precision
plt.figure(figsize=(8, 5))
plt.plot(history.history["precision"], label="train")
plt.plot(history.history["val_precision"], label="val")
plt.title("Precision")
plt.xlabel("Época")
plt.ylim([0, 1])
plt.legend()
plt.tight_layout()
plt.show()

# Recall
plt.figure(figsize=(8, 5))
plt.plot(history.history["recall"], label="train")
plt.plot(history.history["val_recall"], label="val")
plt.title("Recall")
plt.xlabel("Época")
plt.ylim([0, 1])
plt.legend()
plt.tight_layout()
plt.show()

# Accuracy
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="val")
plt.title("Accuracy")
plt.xlabel("Época")
plt.ylim([0, 1])
plt.legend()
plt.tight_layout()
plt.show()

# Final summary
best_acc_idx = int(np.argmax(history.history["val_accuracy"]))
best_acc     = history.history["val_accuracy"][best_acc_idx]

print("=" * 40)
print("RESUMO FINAL")
print("=" * 40)
print(f"Épocas executadas: {n_epochs_run} / {EPOCHS}")
print(f"Melhor val_loss:   {best_val_loss:.4f}")
print(f"Melhor val_auc:    {best_val_auc:.4f}")
print(f"Melhor Accuracy:   {best_acc:.4f} (época {best_acc_idx + 1})")
print("=" * 40)